# 03 — Front-view instant center & roll center

**Concepts (Carroll Smith, "Tune to Win" ch. 3–4 refresher):**

The **instant center (IC)** is where the upper and lower A-arm lines intersect
in the front view. It's the instantaneous pivot point of the wheel relative to
the chassis. As the wheel travels through bump and rebound, the IC moves — and
how it moves determines the character of the suspension.

The **roll center (RC)** is where the line from the contact patch through the IC
crosses the vehicle centerline. It's the point about which the chassis rolls
relative to the ground. RC height determines the split between:
- **Geometric weight transfer** (through the links, instant, no spring compression)
- **Elastic weight transfer** (through the springs, progressive, causes body roll)

The **front-view swing arm length (FVSA)** is the horizontal distance from the
contact patch to the IC. It controls camber gain rate:
- Short FVSA → fast camber change → good for keeping tire flat under roll
- Long FVSA → slow camber change → more predictable but less camber recovery

**What we want for a track-biased street car:**
- RC height: 1–3 inches (25–75mm) above ground, relatively stable through travel
- FVSA that gives about 0.5–1.0° camber gain per inch of bump travel
- IC that doesn't move wildly — smooth, predictable behavior

In [ ]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import matplotlib.pyplot as plt

from suspension.hardpoints import example_frontend_hardpoints
from suspension.kinematics_front import solve_corner, compute_camber, wheel_travel
from suspension.roll_center import (
    front_view_instant_center,
    roll_center_height,
    front_view_swing_arm_length,
    compute_roll_geometry_sweep,
)
from suspension.hardpoint_io import to_dict, write_csv, print_summary

print("Imports OK")

## Load hardpoints and inspect

Using the same placeholder hardpoints as notebook 02. Once you have the
3D scanner data, you'll swap these out by editing the CSV file.

In [ ]:
hp = example_frontend_hardpoints()
print_summary(hp)

# Also save a CSV so you can see the format — this is what you'll edit
# or generate from Onshape later.
write_csv(hp, '../measurements/front_left_placeholder.csv')
print("\nCSV written to ../measurements/front_left_placeholder.csv")
print("Open it in Excel or a text editor to see the format.")

## Run the kinematic sweep and compute roll geometry

In [ ]:
# Sweep +/- 10 degrees (same as notebook 02)
angles_deg = np.linspace(-10, 10, 81)  # finer resolution for smoother curves
angles_rad = np.deg2rad(angles_deg)

# Run the corner solver at each position
results = []
camber = []
travel = []

for angle in angles_rad:
    r = solve_corner(hp, angle)
    results.append(r)
    camber.append(compute_camber(r['wheel_center'], r['contact_patch']))
    travel.append(wheel_travel(hp, r) * 1000)  # mm

camber = np.array(camber)
travel = np.array(travel)

# Compute IC, RC, FVSA at every position
roll_geom = compute_roll_geometry_sweep(hp, results, angles_rad)

print(f"Wheel travel range: {travel.min():.1f} to {travel.max():.1f} mm")
print(f"Camber range:       {camber.min():.2f} to {camber.max():.2f} deg")
print(f"\nRoll center height at static: {roll_geom['rc_z'][len(results)//2]*1000:.1f} mm")
print(f"Roll center range:  {np.nanmin(roll_geom['rc_z'])*1000:.1f} to {np.nanmax(roll_geom['rc_z'])*1000:.1f} mm")
print(f"\nFVSA at static:     {roll_geom['fvsa'][len(results)//2]*1000:.0f} mm")
print(f"FVSA range:         {np.nanmin(roll_geom['fvsa'])*1000:.0f} to {np.nanmax(roll_geom['fvsa'])*1000:.0f} mm")

## Plot 1: The big three curves

These three curves — camber, roll center height, and FVSA, all vs. wheel
travel — are the primary outputs of a front-view kinematic analysis. Every
suspension design discussion starts here.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Camber vs travel
ax = axes[0]
ax.plot(camber, travel, 'b-', linewidth=2)
ax.axhline(0, color='k', linewidth=0.5)
ax.axvline(0, color='k', linewidth=0.5)
ax.set_xlabel('Camber (deg, negative = top in)')
ax.set_ylabel('Wheel travel (mm, + = bump)')
ax.set_title('Camber vs travel')
ax.grid(True, alpha=0.3)

# 2. Roll center height vs travel
ax = axes[1]
ax.plot(roll_geom['rc_z'] * 1000, travel, 'r-', linewidth=2)
ax.axhline(0, color='k', linewidth=0.5)
ax.set_xlabel('Roll center height (mm)')
ax.set_ylabel('Wheel travel (mm, + = bump)')
ax.set_title('Roll center height vs travel')
ax.grid(True, alpha=0.3)

# 3. FVSA vs travel
ax = axes[2]
ax.plot(roll_geom['fvsa'] * 1000, travel, 'g-', linewidth=2)
ax.axhline(0, color='k', linewidth=0.5)
ax.set_xlabel('FVSA (mm)')
ax.set_ylabel('Wheel travel (mm, + = bump)')
ax.set_title('Front-view swing arm length vs travel')
ax.grid(True, alpha=0.3)

fig.suptitle('Front-view kinematic analysis — placeholder hardpoints', fontsize=14)
fig.tight_layout()
plt.show()

## Plot 2: Instant center migration

This plot shows where the IC is in the front view at each travel position.
Ideally the IC moves smoothly and stays in a reasonable region. If it jumps
around wildly, the suspension behavior will be unpredictable.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

valid = roll_geom['valid']
ic_y_mm = roll_geom['ic_y'][valid] * 1000
ic_z_mm = roll_geom['ic_z'][valid] * 1000
travel_valid = travel[valid]

# Color the IC path by travel position
scatter = ax.scatter(ic_y_mm, ic_z_mm, c=travel_valid, cmap='coolwarm',
                     s=20, zorder=3)
plt.colorbar(scatter, ax=ax, label='Wheel travel (mm)')

# Mark static position
mid = len(results) // 2
if valid[mid]:
    ax.plot(roll_geom['ic_y'][mid]*1000, roll_geom['ic_z'][mid]*1000,
            'ko', markersize=12, label='Static IC', zorder=4)

# Draw the ball joints at static to show scale
ax.plot(hp.upper_ball_joint[1]*1000, hp.upper_ball_joint[2]*1000,
        'r^', markersize=8, label='Upper BJ (static)')
ax.plot(hp.lower_ball_joint[1]*1000, hp.lower_ball_joint[2]*1000,
        'bv', markersize=8, label='Lower BJ (static)')

# Draw the arm lines at static to show how they converge at the IC
upper_pc = hp.upper_pivot_center()
lower_pc = hp.lower_pivot_center()
# Extend the lines past the ball joints toward the IC
if valid[mid]:
    ic_static = np.array([roll_geom['ic_y'][mid]*1000, roll_geom['ic_z'][mid]*1000])
    ax.plot([upper_pc[1]*1000, ic_static[0]], [upper_pc[2]*1000, ic_static[1]],
            'r--', alpha=0.4, linewidth=1)
    ax.plot([lower_pc[1]*1000, ic_static[0]], [lower_pc[2]*1000, ic_static[1]],
            'b--', alpha=0.4, linewidth=1)

# Contact patch and RC line
cp = hp.contact_patch
if valid[mid]:
    rc_h = roll_geom['rc_z'][mid] * 1000
    ax.plot([0, cp[1]*1000], [rc_h, cp[2]*1000], 'k-', alpha=0.3,
            linewidth=1, label=f'RC line (RC height = {rc_h:.1f} mm)')
    ax.plot(0, rc_h, 'k*', markersize=15, label='Roll center', zorder=4)

ax.set_xlabel('Y (mm) — lateral')
ax.set_ylabel('Z (mm) — vertical')
ax.set_title('Instant center migration, front view')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
plt.show()

## Plot 3: Front-view linkage with IC and RC

Putting it all together — the suspension at static, with the IC and RC
construction lines visible. This is the classic diagram from every
suspension textbook.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

# Draw the static corner
result_static = solve_corner(hp, 0.0)
upper_pc = hp.upper_pivot_center()
lower_pc = hp.lower_pivot_center()

# A-arms
ax.plot([upper_pc[1]*1000, result_static['upper_ball_joint'][1]*1000],
        [upper_pc[2]*1000, result_static['upper_ball_joint'][2]*1000],
        'b-', linewidth=3, label='Upper A-arm')
ax.plot([lower_pc[1]*1000, result_static['lower_ball_joint'][1]*1000],
        [lower_pc[2]*1000, result_static['lower_ball_joint'][2]*1000],
        'r-', linewidth=3, label='Lower A-arm')

# Upright
ax.plot([result_static['lower_ball_joint'][1]*1000, result_static['upper_ball_joint'][1]*1000],
        [result_static['lower_ball_joint'][2]*1000, result_static['upper_ball_joint'][2]*1000],
        'k-', linewidth=3, label='Upright')

# Wheel
wc = result_static['wheel_center']
cp = result_static['contact_patch']
tire_top = wc + (wc - cp)
ax.plot([cp[1]*1000, tire_top[1]*1000], [cp[2]*1000, tire_top[2]*1000],
        color='gray', linewidth=6, alpha=0.3, label='Wheel/tire')

# IC construction: extend A-arm lines to intersection
ic = front_view_instant_center(upper_pc, result_static['upper_ball_joint'],
                                lower_pc, result_static['lower_ball_joint'])
if ic is not None:
    # Dashed lines from pivot centers through ball joints to IC
    ax.plot([upper_pc[1]*1000, ic[0]*1000], [upper_pc[2]*1000, ic[1]*1000],
            'b--', alpha=0.4, linewidth=1)
    ax.plot([lower_pc[1]*1000, ic[0]*1000], [lower_pc[2]*1000, ic[1]*1000],
            'r--', alpha=0.4, linewidth=1)
    ax.plot(ic[0]*1000, ic[1]*1000, 'ko', markersize=10,
            label=f'Instant center ({ic[0]*1000:.0f}, {ic[1]*1000:.0f}) mm')

    # RC construction: line from contact patch through IC to centerline
    rc_h = roll_center_height(ic, cp)
    ax.plot([0, cp[1]*1000], [rc_h*1000, cp[2]*1000],
            'g--', linewidth=1.5, alpha=0.6)
    ax.plot(0, rc_h*1000, 'g*', markersize=18,
            label=f'Roll center (height = {rc_h*1000:.1f} mm)')

# Pivot center markers
ax.plot(upper_pc[1]*1000, upper_pc[2]*1000, 'bs', markersize=8)
ax.plot(lower_pc[1]*1000, lower_pc[2]*1000, 'rs', markersize=8)

# Ground line and centerline
ax.axhline(0, color='saddlebrown', linewidth=1, alpha=0.5, label='Ground')
ax.axvline(0, color='gray', linewidth=1, alpha=0.3, linestyle=':', label='Centerline')

ax.set_xlabel('Y (mm) — lateral')
ax.set_ylabel('Z (mm) — vertical')
ax.set_title('Front-view suspension geometry with IC and RC construction')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.2)
ax.set_aspect('equal')
plt.show()